# Temporal validation, calibration and uncertainty

Reproducible historical research notebook using source utilities. This published version removes cached outputs and order/customer identifiers. Download the three required Olist CSVs from the original Kaggle dataset before execution.

**Scope:** binary late delivery among confirmed delivered orders. The retrospective holdout has already been inspected during development; results must not be represented as newly untouched validation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from src.train import create_snapshot, aggregate_order_items, build_features, make_model, FEATURE_COLUMNS

DATA = Path("data/raw")
orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")
items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
item_features = aggregate_order_items(items)
print("Orders in source dataset:", len(orders))

## Historical cohorts

Training: Jan–Oct 2017, delivery observed before 1 Jan 2018. Calibration: November 2017, observed before 1 Jan 2018. Validation: Jan–Mar 2018, observed before 1 Jul 2018. Retrospective test: July–Aug 2018 with recorded delivery in the provided historical extract. The final test cutoff below is a computational convenience, **not** evidence that data were available prospectively in 2019.

In [ ]:
train_orders = create_snapshot(orders, "2017-01-01", "2017-11-01", "2018-01-01")
cal_orders = create_snapshot(orders, "2017-11-01", "2017-12-01", "2018-01-01")
val_orders = create_snapshot(orders, "2018-01-01", "2018-04-01", "2018-07-01")
test_orders = create_snapshot(orders, "2018-07-01", "2018-09-01", "2019-01-01")
sets = {
    "Training": build_features(train_orders, customers, item_features),
    "Calibration": build_features(cal_orders, customers, item_features),
    "Validation": build_features(val_orders, customers, item_features),
    "Retrospective test": build_features(test_orders, customers, item_features),
}
print(pd.DataFrame([{
    "set": name,
    "orders": len(df),
    "late_rate": round(df.late_delivery.mean(), 4)
} for name, df in sets.items()]).to_string(index=False))
assert all(
    set(a.order_id).isdisjoint(b.order_id)
    for idx, a in enumerate(sets.values())
    for b in list(sets.values())[idx+1:]
)
X_train, y_train = sets["Training"][FEATURE_COLUMNS], sets["Training"].late_delivery
X_cal, y_cal = sets["Calibration"][FEATURE_COLUMNS], sets["Calibration"].late_delivery
X_val, y_val = sets["Validation"][FEATURE_COLUMNS], sets["Validation"].late_delivery
X_test, y_test = sets["Retrospective test"][FEATURE_COLUMNS], sets["Retrospective test"].late_delivery

## Classifier comparison on validation

Each non-dummy classifier has the same preprocessor and feature schema. No test observations are used to fit or tune the models. Selection is exploratory because validation has been inspected repeatedly.

In [ ]:
models = {
    "Dummy Baseline": clone(make_model()).set_params(classifier=DummyClassifier(strategy="prior")),
    "Logistic Regression": clone(make_model()).set_params(classifier=LogisticRegression(max_iter=2000, random_state=42)),
    "Random Forest": clone(make_model()).set_params(classifier=RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=20, random_state=42, n_jobs=-1
    )),
    "XGBoost": make_model(),
}

def scores(y, p):
    return {
        "ROC-AUC": roc_auc_score(y, p),
        "Average Precision": average_precision_score(y, p),
        "Brier Score": brier_score_loss(y, p),
        "Mean Predicted (%)": p.mean() * 100,
    }

results = []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    results.append({"Model": name, **scores(y_val, pipeline.predict_proba(X_val)[:, 1])})
display(pd.DataFrame(results).round(4))

## Risk prioritization under limited review capacity

Top-10% precision and recall measure selection concentration in the observed validation cohort. They do not estimate the *causal benefit* of intervening.

In [ ]:
def at_capacity(y, p, fraction=0.10):
    actual = np.asarray(y)
    k = max(1, int(np.ceil(fraction * len(actual))))
    selected = np.argsort(-np.asarray(p), kind="stable")[:k]
    precision = actual[selected].mean()
    recall = actual[selected].sum() / actual.sum()
    return {"Selected": k, "Precision@10%": precision, "Recall@10%": recall,
            "Lift@10%": precision / actual.mean()}

display(pd.DataFrame([
    {"Model": name, **at_capacity(y_val, model.predict_proba(X_val)[:, 1])}
    for name, model in models.items()
]).round(4))

## Temporal diagnostics

High-ranked orders may cluster differently each month. Observe actual event prevalence and average model scores separately; matching averages alone does not establish calibration.

In [ ]:
xgb = models["XGBoost"]
val_monthly = sets["Validation"][["order_purchase_timestamp", "late_delivery"]].copy()
val_monthly["score"] = xgb.predict_proba(X_val)[:, 1]
val_monthly["month"] = val_monthly.order_purchase_timestamp.dt.to_period("M")
monthly = val_monthly.groupby("month").agg(
    orders=("late_delivery", "size"),
    observed_rate=("late_delivery", "mean"),
    mean_score=("score", "mean")
)
display(monthly.round(4))

## Sigmoid calibration (held-out calibration cohort)

FrozenEstimator preserves the already fitted XGBoost. Fit calibration only on the separate November cohort, then compare on validation. Calibration may be unstable under changing event prevalence.

In [ ]:
calibrated = CalibratedClassifierCV(
    estimator=FrozenEstimator(xgb), method="sigmoid"
)
calibrated.fit(X_cal, y_cal)
val_results = []
for name, m in [("Original XGBoost", xgb), ("Calibrated XGBoost", calibrated)]:
    val_results.append({"Model": name, **scores(y_val, m.predict_proba(X_val)[:, 1])})
display(pd.DataFrame(val_results).round(4))

## Retrospective holdout evaluation

Use this already-inspected historical holdout for descriptive reporting only. The delivered-status filter excludes unconfirmed outcomes, and late outcomes near the dataset boundary may be censored.

In [ ]:
test_result = []
for name, m in [("Original XGBoost", xgb), ("Calibrated XGBoost", calibrated)]:
    p = m.predict_proba(X_test)[:, 1]
    test_result.append({"Model": name, **scores(y_test, p), **at_capacity(y_test, p)})
test_result_df = pd.DataFrame(test_result)
display(test_result_df.round(4))

## Bootstrap variability of holdout *performance*

These approximate percentile intervals treat sampled orders as independent. They do not measure individual prediction uncertainty, seller clustering, future distribution shift, or model training variability.

In [ ]:
y = y_test.to_numpy()
p = xgb.predict_proba(X_test)[:, 1]
rng = np.random.default_rng(42)
original = {**scores(y, p), **at_capacity(y, p)}
metrics_to_report = ["ROC-AUC", "Average Precision", "Brier Score", "Precision@10%"]

replicates = []
for _ in range(1000):
    idx = rng.integers(0, len(y), len(y))
    yy, pp = y[idx], p[idx]
    if len(np.unique(yy)) < 2:
        continue
    replicates.append({**scores(yy, pp), **at_capacity(yy, pp)})

boot = pd.DataFrame(replicates)
summary = pd.DataFrame([
    {"Metric": key, "Estimate": original[key],
     "Lower 95%": boot[key].quantile(0.025),
     "Upper 95%": boot[key].quantile(0.975)}
    for key in metrics_to_report
])
display(summary.round(4))

## Reporting limitations

Historical results are retrospective, targeted only to confirmed delivered orders, and lack a fully prospectively observed outcome-availability snapshot. Repeated validation and prior test inspection affect the independence of model selection. A production system would require prospective evaluation, outcome completeness, seller-clustered/time-block uncertainty analysis, shift monitoring, and evidence of actual intervention benefit.